In [43]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import gymnasium as gym
from gymnasium import Env, spaces
import torch
import torch.nn as nn
import torch.optim as optim
import random
import seaborn as sns
import time
from collections import deque

## Model 1: Base Model

In [44]:

##  MountainCar environment

class CustomMountainCarContinuous(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        # Action space: continuous force in [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        # Observation space: position [-1.2, 0.6], velocity [-0.07, 0.07], but not discrete
        self.observation_space = spaces.Box(
            low=np.array([-1.2, -0.07]), high=np.array([0.6, 0.07]), dtype=np.float32
        )
        # Constants
        self.force_multiplier = 0.0015
        self.gravity = 0.0025
        self.max_steps = 200
        self.state = None
        self.step_count = 0
        # For visualization using Gym's renderer
        self.render_mode = render_mode
        self.viewer_env = gym.make("MountainCarContinuous-v0", render_mode=render_mode) if render_mode else None
    
    def step(self, action):
        assert self.action_space.contains(action), "Invalid action"
        action = action[0]  # Extract scalar force F
        pos, vel = self.state
        self.step_count += 1
        
        # Equations of motion
        vel_next = vel + action * self.force_multiplier - np.cos(3 * pos) * self.gravity
        pos_next = pos + vel_next
        
        # Boundary conditions (walls)
        if pos_next < -1.2:
            pos_next = -1.2
            vel_next = 0
        elif pos_next > 0.6:
            pos_next = 0.6
            vel_next = 0
        
        self.state = [pos_next, vel_next]
        
        # Termination conditions
        terminated = pos_next >= 0.5
        truncated = self.step_count >= self.max_steps
        reward = -1.0 if not terminated else 0.0  # Standard reward
        
        # Render if enabled
        if self.viewer_env:
            self.viewer_env.unwrapped.state = np.array([pos_next, vel_next], dtype=np.float32)
            self.viewer_env.render()
        
        return np.array(self.state, dtype=np.float32), reward, terminated, truncated, {}
    
    def reset(self, seed=None):
        # Initial state: position random in [-0.6, -0.4], velocity=0
        self.state = [np.random.uniform(-0.6, -0.4), 0.0] ## Uniform Distribution
        self.step_count = 0
        if self.viewer_env:
            self.viewer_env.reset(seed=seed)
            self.viewer_env.unwrapped.state = np.array(self.state, dtype=np.float32)
        return np.array(self.state, dtype=np.float32), {}
    
    def close(self):
        if self.viewer_env:
            self.viewer_env.close()

In [45]:
## DQN Setup with State Normalization
import torch


class DQN(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(DQN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim)
        )
    
    def forward(self, x):
        return self.net(x)

# Parameters
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.1
n_episodes = 1000
max_steps = 200
batch_size = 64
memory_capacity = 10000
target_update = 10

# Action discretization
num_actions = 10
discrete_actions = np.linspace(-1.0, 1.0, num_actions)

# State normalization parameters --> THIS IS NEEDED IN ORDER FOR THE MODEL TO TRAIN PROPERLY.
state_mean = np.array([(-1.2 + 0.6) / 2, 0.0])  
state_scale = np.array([(0.6 - (-1.2)) / 2, 0.07]) 

# Initialize
env = CustomMountainCarContinuous()
input_dim = 2
output_dim = num_actions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_network = DQN(input_dim, output_dim).to(device)
target_network = DQN(input_dim, output_dim).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.Adam(q_network.parameters(), lr=0.001)
criterion = nn.MSELoss()
memory = deque(maxlen=memory_capacity)

In [46]:
## DQN Training Loop with Debugging to see where the problem is 
episode_rewards = []
episode_steps = []
successes = []

start_time = time.time()

for episode in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < max_steps:
        # Normalize state
        norm_state = (state - state_mean) / state_scale
        state_tensor = torch.FloatTensor(norm_state).to(device)
        
        # Epsilon-greedy
        if random.random() < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            with torch.no_grad():
                q_values = q_network(state_tensor)
                action_idx = q_values.argmax().item()
        
        action = np.array([discrete_actions[action_idx]], dtype=np.float32)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
        
        # Store normalized state
        norm_next_state = (next_state - state_mean) / state_scale
        memory.append((norm_state, action_idx, reward, norm_next_state, terminated))
        state = next_state
        done = terminated or truncated
        
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_network(next_states).max(1)[0]
                targets = rewards + gamma * next_q_values * (1 - dones)
            
            loss = criterion(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    successes.append(1 if terminated else 0)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % target_update == 0:
        target_network.load_state_dict(q_network.state_dict())
    
    # Debugging every 100 episodes
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}: Reward={total_reward:.2f}, Steps={steps}, Epsilon={epsilon:.3f}")

training_time = time.time() - start_time
env.close()

/tmp/ipykernel_5652/3774966266.py:47: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  dones = torch.FloatTensor(dones).to(device)


Episode 100: Reward=-200.00, Steps=200, Epsilon=0.606
Episode 200: Reward=-200.00, Steps=200, Epsilon=0.367
Episode 300: Reward=-200.00, Steps=200, Epsilon=0.222
Episode 400: Reward=-200.00, Steps=200, Epsilon=0.135
Episode 500: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 600: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 700: Reward=-132.00, Steps=133, Epsilon=0.100
Episode 800: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 900: Reward=-140.00, Steps=141, Epsilon=0.100
Episode 1000: Reward=-200.00, Steps=200, Epsilon=0.100


In [47]:
## Evaluations and Metrics
avg_reward_last_100 = np.mean(episode_rewards[-100:])
success_rate_last_100 = np.mean(successes[-100:]) * 100
avg_steps_last_100 = np.mean([
    s if succ else max_steps for s, succ in zip(episode_steps[-100:], successes[-100:])
])

print(f"Training time: {training_time:.2f} seconds")
print(f"Average reward (last 100 episodes): {avg_reward_last_100:.2f}")
print(f"Success rate (last 100 episodes): {success_rate_last_100:.2f}%")
print(f"Average steps (last 100 episodes, max if fail): {avg_steps_last_100:.2f}")

Training time: 321.38 seconds
Average reward (last 100 episodes): -166.79
Success rate (last 100 episodes): 67.00%
Average steps (last 100 episodes, max if fail): 167.46


## Model 2: Optimizer = RMSprop

In [48]:
# Initialize
env = CustomMountainCarContinuous()
input_dim = 2
output_dim = num_actions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_network = DQN(input_dim, output_dim).to(device)
target_network = DQN(input_dim, output_dim).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.RMSprop(q_network.parameters(), lr=0.001)  ## Different
criterion = nn.MSELoss()
memory = deque(maxlen=memory_capacity)

In [49]:
## DQN Training Loop with Debugging to see where the problem is 
episode_rewards = []
episode_steps = []
successes = []

start_time = time.time()

for episode in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < max_steps:
        # Normalize state
        norm_state = (state - state_mean) / state_scale
        state_tensor = torch.FloatTensor(norm_state).to(device)
        
        # Epsilon-greedy
        if random.random() < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            with torch.no_grad():
                q_values = q_network(state_tensor)
                action_idx = q_values.argmax().item()
        
        action = np.array([discrete_actions[action_idx]], dtype=np.float32)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
        
        # Store normalized state
        norm_next_state = (next_state - state_mean) / state_scale
        memory.append((norm_state, action_idx, reward, norm_next_state, terminated))
        state = next_state
        done = terminated or truncated
        
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_network(next_states).max(1)[0]
                targets = rewards + gamma * next_q_values * (1 - dones)
            
            loss = criterion(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    successes.append(1 if terminated else 0)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % target_update == 0:
        target_network.load_state_dict(q_network.state_dict())
    
    # Debugging every 100 episodes
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}: Reward={total_reward:.2f}, Steps={steps}, Epsilon={epsilon:.3f}")

training_time = time.time() - start_time
env.close()

/tmp/ipykernel_5652/3774966266.py:47: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  dones = torch.FloatTensor(dones).to(device)


Episode 100: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 200: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 300: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 400: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 500: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 600: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 700: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 800: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 900: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 1000: Reward=-200.00, Steps=200, Epsilon=0.100


In [50]:
## Evaluations and Metrics
avg_reward_last_100 = np.mean(episode_rewards[-100:])
success_rate_last_100 = np.mean(successes[-100:]) * 100
avg_steps_last_100 = np.mean([
    s if succ else max_steps for s, succ in zip(episode_steps[-100:], successes[-100:])
])

print(f"Training time: {training_time:.2f} seconds")
print(f"Average reward (last 100 episodes): {avg_reward_last_100:.2f}")
print(f"Success rate (last 100 episodes): {success_rate_last_100:.2f}%")
print(f"Average steps (last 100 episodes, max if fail): {avg_steps_last_100:.2f}")

Training time: 302.27 seconds
Average reward (last 100 episodes): -200.00
Success rate (last 100 episodes): 0.00%
Average steps (last 100 episodes, max if fail): 200.00


## Model 3 : Loss function = Huber

In [51]:
# Initialize
env = CustomMountainCarContinuous()
input_dim = 2
output_dim = num_actions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_network = DQN(input_dim, output_dim).to(device)
target_network = DQN(input_dim, output_dim).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.Adam(q_network.parameters(), lr=0.001)
criterion = nn.HuberLoss()
memory = deque(maxlen=memory_capacity)

In [52]:
## DQN Training Loop with Debugging to see where the problem is 
episode_rewards = []
episode_steps = []
successes = []

start_time = time.time()

for episode in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < max_steps:
        # Normalize state
        norm_state = (state - state_mean) / state_scale
        state_tensor = torch.FloatTensor(norm_state).to(device)
        
        # Epsilon-greedy
        if random.random() < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            with torch.no_grad():
                q_values = q_network(state_tensor)
                action_idx = q_values.argmax().item()
        
        action = np.array([discrete_actions[action_idx]], dtype=np.float32)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
        
        # Store normalized state
        norm_next_state = (next_state - state_mean) / state_scale
        memory.append((norm_state, action_idx, reward, norm_next_state, terminated))
        state = next_state
        done = terminated or truncated
        
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_network(next_states).max(1)[0]
                targets = rewards + gamma * next_q_values * (1 - dones)
            
            loss = criterion(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    successes.append(1 if terminated else 0)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % target_update == 0:
        target_network.load_state_dict(q_network.state_dict())
    
    # Debugging every 100 episodes
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}: Reward={total_reward:.2f}, Steps={steps}, Epsilon={epsilon:.3f}")

training_time = time.time() - start_time
env.close()

/tmp/ipykernel_5652/3774966266.py:47: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  dones = torch.FloatTensor(dones).to(device)


Episode 100: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 200: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 300: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 400: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 500: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 600: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 700: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 800: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 900: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 1000: Reward=-200.00, Steps=200, Epsilon=0.100


In [53]:
## Evaluations and Metrics
avg_reward_last_100 = np.mean(episode_rewards[-100:])
success_rate_last_100 = np.mean(successes[-100:]) * 100
avg_steps_last_100 = np.mean([
    s if succ else max_steps for s, succ in zip(episode_steps[-100:], successes[-100:])
])

print(f"Training time: {training_time:.2f} seconds")
print(f"Average reward (last 100 episodes): {avg_reward_last_100:.2f}")
print(f"Success rate (last 100 episodes): {success_rate_last_100:.2f}%")
print(f"Average steps (last 100 episodes, max if fail): {avg_steps_last_100:.2f}")

Training time: 336.51 seconds
Average reward (last 100 episodes): -200.00
Success rate (last 100 episodes): 0.00%
Average steps (last 100 episodes, max if fail): 200.00


## Model 4: Reward Function = -1 for each 3 steps not at goal

In [54]:

##  MountainCar environment

class CustomMountainCarContinuous(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        # Action space: continuous force in [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        # Observation space: position [-1.2, 0.6], velocity [-0.07, 0.07], but not discrete
        self.observation_space = spaces.Box(
            low=np.array([-1.2, -0.07]), high=np.array([0.6, 0.07]), dtype=np.float32
        )
        # Constants
        self.force_multiplier = 0.0015
        self.gravity = 0.0025
        self.max_steps = 200
        self.state = None
        self.step_count = 0
        # For visualization using Gym's renderer
        self.render_mode = render_mode
        self.viewer_env = gym.make("MountainCarContinuous-v0", render_mode=render_mode) if render_mode else None
    
    def step(self, action):
        assert self.action_space.contains(action), "Invalid action"
        action = action[0]  # Extract scalar force F
        pos, vel = self.state
        self.step_count += 1
        
        # Equations of motion
        vel_next = vel + action * self.force_multiplier - np.cos(3 * pos) * self.gravity
        pos_next = pos + vel_next
        
        # Boundary conditions (walls)
        if pos_next < -1.2:
            pos_next = -1.2
            vel_next = 0
        elif pos_next > 0.6:
            pos_next = 0.6
            vel_next = 0
        
        self.state = [pos_next, vel_next]
        
        # Termination conditions
        terminated = pos_next >= 0.5
        truncated = self.step_count >= self.max_steps
        reward = -1.0 if self.step_count % 3 == 0 and not terminated else 0.0 ### reward = -1, if not reached at the target each 3 steps
        
        # Render if enabled
        if self.viewer_env:
            self.viewer_env.unwrapped.state = np.array([pos_next, vel_next], dtype=np.float32)
            self.viewer_env.render()
        
        return np.array(self.state, dtype=np.float32), reward, terminated, truncated, {}
    
    def reset(self, seed=None):
        # Initial state: position random in [-0.6, -0.4], velocity=0
        self.state = [np.random.uniform(-0.6, -0.4), 0.0] ## Uniform Distribution
        self.step_count = 0
        if self.viewer_env:
            self.viewer_env.reset(seed=seed)
            self.viewer_env.unwrapped.state = np.array(self.state, dtype=np.float32)
        return np.array(self.state, dtype=np.float32), {}
    
    def close(self):
        if self.viewer_env:
            self.viewer_env.close()

In [55]:
# Initialize
env = CustomMountainCarContinuous()
input_dim = 2
output_dim = num_actions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_network = DQN(input_dim, output_dim).to(device)
target_network = DQN(input_dim, output_dim).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.Adam(q_network.parameters(), lr=0.001)
criterion = nn.MSELoss()
memory = deque(maxlen=memory_capacity)

In [56]:
## DQN Training Loop with Debugging to see where the problem is 
episode_rewards = []
episode_steps = []
successes = []

start_time = time.time()

for episode in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < max_steps:
        # Normalize state
        norm_state = (state - state_mean) / state_scale
        state_tensor = torch.FloatTensor(norm_state).to(device)
        
        # Epsilon-greedy
        if random.random() < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            with torch.no_grad():
                q_values = q_network(state_tensor)
                action_idx = q_values.argmax().item()
        
        action = np.array([discrete_actions[action_idx]], dtype=np.float32)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
        
        # Store normalized state
        norm_next_state = (next_state - state_mean) / state_scale
        memory.append((norm_state, action_idx, reward, norm_next_state, terminated))
        state = next_state
        done = terminated or truncated
        
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_network(next_states).max(1)[0]
                targets = rewards + gamma * next_q_values * (1 - dones)
            
            loss = criterion(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    successes.append(1 if terminated else 0)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % target_update == 0:
        target_network.load_state_dict(q_network.state_dict())
    
    # Debugging every 100 episodes
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}: Reward={total_reward:.2f}, Steps={steps}, Epsilon={epsilon:.3f}")

training_time = time.time() - start_time
env.close()

/tmp/ipykernel_5652/3774966266.py:47: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  dones = torch.FloatTensor(dones).to(device)


Episode 100: Reward=-66.00, Steps=200, Epsilon=0.100
Episode 200: Reward=-42.00, Steps=127, Epsilon=0.100
Episode 300: Reward=-66.00, Steps=200, Epsilon=0.100
Episode 400: Reward=-55.00, Steps=167, Epsilon=0.100
Episode 500: Reward=-55.00, Steps=168, Epsilon=0.100
Episode 600: Reward=-66.00, Steps=200, Epsilon=0.100
Episode 700: Reward=-55.00, Steps=167, Epsilon=0.100
Episode 800: Reward=-58.00, Steps=177, Epsilon=0.100
Episode 900: Reward=-47.00, Steps=144, Epsilon=0.100
Episode 1000: Reward=-34.00, Steps=103, Epsilon=0.100


In [57]:
## Evaluations and Metrics
avg_reward_last_100 = np.mean(episode_rewards[-100:])
success_rate_last_100 = np.mean(successes[-100:]) * 100
avg_steps_last_100 = np.mean([
    s if succ else max_steps for s, succ in zip(episode_steps[-100:], successes[-100:])
])

print(f"Training time: {training_time:.2f} seconds")
print(f"Average reward (last 100 episodes): {avg_reward_last_100:.2f}")
print(f"Success rate (last 100 episodes): {success_rate_last_100:.2f}%")
print(f"Average steps (last 100 episodes, max if fail): {avg_steps_last_100:.2f}")

Training time: 261.99 seconds
Average reward (last 100 episodes): -31.46
Success rate (last 100 episodes): 99.00%
Average steps (last 100 episodes, max if fail): 96.42


## Model 5: Reward = -1 for each step not at goal and +100 if at goal

In [58]:

##  MountainCar environment

class CustomMountainCarContinuous(gym.Env):
    def __init__(self, render_mode=None):
        super().__init__()
        # Action space: continuous force in [-1, 1]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)
        # Observation space: position [-1.2, 0.6], velocity [-0.07, 0.07], but not discrete
        self.observation_space = spaces.Box(
            low=np.array([-1.2, -0.07]), high=np.array([0.6, 0.07]), dtype=np.float32
        )
        # Constants
        self.force_multiplier = 0.0015
        self.gravity = 0.0025
        self.max_steps = 200
        self.state = None
        self.step_count = 0
        # For visualization using Gym's renderer
        self.render_mode = render_mode
        self.viewer_env = gym.make("MountainCarContinuous-v0", render_mode=render_mode) if render_mode else None
    
    def step(self, action):
        assert self.action_space.contains(action), "Invalid action"
        action = action[0]  # Extract scalar force F
        pos, vel = self.state
        self.step_count += 1
        
        # Equations of motion
        vel_next = vel + action * self.force_multiplier - np.cos(3 * pos) * self.gravity
        pos_next = pos + vel_next
        
        # Boundary conditions (walls)
        if pos_next < -1.2:
            pos_next = -1.2
            vel_next = 0
        elif pos_next > 0.6:
            pos_next = 0.6
            vel_next = 0
        
        self.state = [pos_next, vel_next]
        
        # Termination conditions
        terminated = pos_next >= 0.5
        truncated = self.step_count >= self.max_steps
        reward = -1.0 if not terminated else 100  
        
        # Render if enabled
        if self.viewer_env:
            self.viewer_env.unwrapped.state = np.array([pos_next, vel_next], dtype=np.float32)
            self.viewer_env.render()
        
        return np.array(self.state, dtype=np.float32), reward, terminated, truncated, {}
    
    def reset(self, seed=None):
        # Initial state: position random in [-0.6, -0.4], velocity=0
        self.state = [np.random.uniform(-0.6, -0.4), 0.0] ## Uniform Distribution
        self.step_count = 0
        if self.viewer_env:
            self.viewer_env.reset(seed=seed)
            self.viewer_env.unwrapped.state = np.array(self.state, dtype=np.float32)
        return np.array(self.state, dtype=np.float32), {}
    
    def close(self):
        if self.viewer_env:
            self.viewer_env.close()

In [59]:
# Initialize
env = CustomMountainCarContinuous()
input_dim = 2
output_dim = num_actions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_network = DQN(input_dim, output_dim).to(device)
target_network = DQN(input_dim, output_dim).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.Adam(q_network.parameters(), lr=0.001)
criterion = nn.MSELoss()
memory = deque(maxlen=memory_capacity)

In [60]:
## DQN Training Loop with Debugging to see where the problem is 
episode_rewards = []
episode_steps = []
successes = []

start_time = time.time()

for episode in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < max_steps:
        # Normalize state
        norm_state = (state - state_mean) / state_scale
        state_tensor = torch.FloatTensor(norm_state).to(device)
        
        # Epsilon-greedy
        if random.random() < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            with torch.no_grad():
                q_values = q_network(state_tensor)
                action_idx = q_values.argmax().item()
        
        action = np.array([discrete_actions[action_idx]], dtype=np.float32)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
        
        # Store normalized state
        norm_next_state = (next_state - state_mean) / state_scale
        memory.append((norm_state, action_idx, reward, norm_next_state, terminated))
        state = next_state
        done = terminated or truncated
        
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_network(next_states).max(1)[0]
                targets = rewards + gamma * next_q_values * (1 - dones)
            
            loss = criterion(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    successes.append(1 if terminated else 0)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % target_update == 0:
        target_network.load_state_dict(q_network.state_dict())
    
    # Debugging every 100 episodes
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}: Reward={total_reward:.2f}, Steps={steps}, Epsilon={epsilon:.3f}")

training_time = time.time() - start_time
env.close()

/tmp/ipykernel_5652/3774966266.py:47: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  dones = torch.FloatTensor(dones).to(device)


Episode 100: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 200: Reward=-55.00, Steps=156, Epsilon=0.100
Episode 300: Reward=-23.00, Steps=124, Epsilon=0.100
Episode 400: Reward=-54.00, Steps=155, Epsilon=0.100
Episode 500: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 600: Reward=28.00, Steps=73, Epsilon=0.100
Episode 700: Reward=20.00, Steps=81, Epsilon=0.100
Episode 800: Reward=16.00, Steps=85, Epsilon=0.100
Episode 900: Reward=18.00, Steps=83, Epsilon=0.100
Episode 1000: Reward=31.00, Steps=70, Epsilon=0.100


In [61]:
## Evaluations and Metrics
avg_reward_last_100 = np.mean(episode_rewards[-100:])
success_rate_last_100 = np.mean(successes[-100:]) * 100
avg_steps_last_100 = np.mean([
    s if succ else max_steps for s, succ in zip(episode_steps[-100:], successes[-100:])
])

print(f"Training time: {training_time:.2f} seconds")
print(f"Average reward (last 100 episodes): {avg_reward_last_100:.2f}")
print(f"Success rate (last 100 episodes): {success_rate_last_100:.2f}%")
print(f"Average steps (last 100 episodes, max if fail): {avg_steps_last_100:.2f}")

Training time: 222.88 seconds
Average reward (last 100 episodes): 16.07
Success rate (last 100 episodes): 98.00%
Average steps (last 100 episodes, max if fail): 82.91


## Model 6: Reward = -1 for each step not at goal and +100 if at goal + Loss fn = huber + optimizer = RMSprop

In [62]:
# Initialize
env = CustomMountainCarContinuous()
input_dim = 2
output_dim = num_actions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
q_network = DQN(input_dim, output_dim).to(device)
target_network = DQN(input_dim, output_dim).to(device)
target_network.load_state_dict(q_network.state_dict())
optimizer = optim.RMSprop(q_network.parameters(), lr=0.001)
criterion = nn.HuberLoss()
memory = deque(maxlen=memory_capacity)

In [63]:
## DQN Training Loop with Debugging to see where the problem is 
episode_rewards = []
episode_steps = []
successes = []

start_time = time.time()

for episode in range(n_episodes):
    state, _ = env.reset()
    total_reward = 0
    steps = 0
    done = False
    
    while not done and steps < max_steps:
        # Normalize state
        norm_state = (state - state_mean) / state_scale
        state_tensor = torch.FloatTensor(norm_state).to(device)
        
        # Epsilon-greedy
        if random.random() < epsilon:
            action_idx = random.randint(0, num_actions - 1)
        else:
            with torch.no_grad():
                q_values = q_network(state_tensor)
                action_idx = q_values.argmax().item()
        
        action = np.array([discrete_actions[action_idx]], dtype=np.float32)
        
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        steps += 1
        
        # Store normalized state
        norm_next_state = (next_state - state_mean) / state_scale
        memory.append((norm_state, action_idx, reward, norm_next_state, terminated))
        state = next_state
        done = terminated or truncated
        
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)
            states, actions, rewards, next_states, dones = zip(*batch)
            
            states = torch.FloatTensor(states).to(device)
            actions = torch.LongTensor(actions).to(device)
            rewards = torch.FloatTensor(rewards).to(device)
            next_states = torch.FloatTensor(next_states).to(device)
            dones = torch.FloatTensor(dones).to(device)
            
            q_values = q_network(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            with torch.no_grad():
                next_q_values = target_network(next_states).max(1)[0]
                targets = rewards + gamma * next_q_values * (1 - dones)
            
            loss = criterion(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    episode_rewards.append(total_reward)
    episode_steps.append(steps)
    successes.append(1 if terminated else 0)
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    
    if (episode + 1) % target_update == 0:
        target_network.load_state_dict(q_network.state_dict())
    
    # Debugging every 100 episodes
    if (episode + 1) % 100 == 0:
        print(f"Episode {episode + 1}: Reward={total_reward:.2f}, Steps={steps}, Epsilon={epsilon:.3f}")

training_time = time.time() - start_time
env.close()

/tmp/ipykernel_5652/3774966266.py:47: DeprecationWarning: In future, it will be an error for 'np.bool_' scalars to be interpreted as an index
  dones = torch.FloatTensor(dones).to(device)


Episode 100: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 200: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 300: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 400: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 500: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 600: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 700: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 800: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 900: Reward=-200.00, Steps=200, Epsilon=0.100
Episode 1000: Reward=-200.00, Steps=200, Epsilon=0.100


In [64]:
## Evaluations and Metrics
avg_reward_last_100 = np.mean(episode_rewards[-100:])
success_rate_last_100 = np.mean(successes[-100:]) * 100
avg_steps_last_100 = np.mean([
    s if succ else max_steps for s, succ in zip(episode_steps[-100:], successes[-100:])
])

print(f"Training time: {training_time:.2f} seconds")
print(f"Average reward (last 100 episodes): {avg_reward_last_100:.2f}")
print(f"Success rate (last 100 episodes): {success_rate_last_100:.2f}%")
print(f"Average steps (last 100 episodes, max if fail): {avg_steps_last_100:.2f}")

Training time: 301.46 seconds
Average reward (last 100 episodes): -200.00
Success rate (last 100 episodes): 0.00%
Average steps (last 100 episodes, max if fail): 200.00
